In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import torchvision.models as models #το models είναι submodule,περιέχει δηλαδή όλα τα μοντέλα και το resnet18()
from sklearn.metrics import confusion_matrix
from PIL import Image
import collections

In [18]:
#Φορτ΄ώνουμε το pre-trained μοντέλο ResNet50 με προεκπαιδευμένα βάρη από το dataset ImageNet.
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
# Αντί για απλό fc layer
#model.fc = nn.Sequential(
#    nn.Dropout(0.5),
#    nn.Linear(512, 3))

In [19]:
#model.fc = nn.Linear(in_features=512, out_features=4) #Το model.fc είναι ένα μέρος του Layer.Είναι σαν να λέμε,πάρε αυτό
model.fc = nn.Linear(in_features=2048, out_features=3) #Από 4 κλάσεις το μοντέλο το ορίζουμε σε 3
#το μέρος του μοντέλου και αντικατέστησέ το με αυτό.

In [20]:
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,hue=0.2,saturation=0.2),  
    transforms.RandomPerspective(distortion_scale=0.15, p=0.5),  #Αυτό ορίζει την γωνία που θα είναι β΄λέπει την εικόνα (0.15) και την πιθανότητα
                                                                 #p οτι το 50% των εικόνων θα έχουν αυτό το transform
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)),  
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

transform_test=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

In [21]:
train_dataset=torchvision.datasets.ImageFolder(root="C:/Users/panag/Desktop/Thesis/Datasets/KMeans_Dataset_with_ResNet50/train",
                                              transform=transform_train)

test_dataset=torchvision.datasets.ImageFolder(root="C:/Users/panag/Desktop/Thesis/Datasets/KMeans_Dataset_with_ResNet50/test",
                                            transform=transform_test)

In [22]:
train_loader=torch.utils.data.DataLoader(dataset=train_dataset,
                                         batch_size=32,num_workers=2,shuffle=True)

test_loader=torch.utils.data.DataLoader(dataset=test_dataset,
                                       batch_size=32,num_workers=2,shuffle=False)

In [23]:
#Ορίζουμε μια μεταβλητή το device η οποία λέει ότι αν υπάρχει η cuda (gpu) να την χρησιμοποιήσει,αλλιώς την cpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [24]:
#Στη συνέχεια πρέπει να περάσουμε την gpu μέσα στο μοντέλο,ώστε σε όλες τις εικόνες να εφαρμωστεί το device
model=model.to(device)
#Αν δεν το εφαρμώσουμε και στο μοντέλο, θα μας εμφανίσει error γιατί οι εικόνες μας θα βρίσκονται σε διαφορετικά σημεία.

In [25]:
#Για να υπολογίσουμε την διαφορά μεταξύ της προβλεπόμενης κατανομής και της πραγματικής χρησιμοποιούμε την Loss Function
#και πιο συγκεκριμένα για 3 κλάσεις που έχουμε την CrossEntropy μέτρηση.
criterion=nn.CrossEntropyLoss()

In [26]:
#Εδ΄ώ αυτό που κάνουμε είναι να μην αφήνουμε το conv1,conv2,layer1, etc να παίρνει biases & weights κατά το back propagation 
#αλλά αντιθέτως να ενημερώνονται μόνο από το fully conected layer κατά το back propagation
for param in model.parameters():
    param.requires_grad=False
for param in model.layer3.parameters():   #Ενεργοποιούμε την ενημέρωση των bias/weights του layer3
    param.requires_grad=True
for parm in model.layer4.parameters():    #Ενεργοποιούμε την ενημέρωση των bias/weights του layer4
    param.requires_grad=True
for param in model.fc.parameters():
    param.requires_grad=True

In [27]:
#Κάνουμε χρήση του αλγορύθμου Adam και ενημερώνουμε τις παραμέτρους μόνο του fc layer ορίζοντας το learning rate που θέλουμε
#optimizer=optim.Adam(model.fc.parameters(),lr=0.001)  #Παλιό optimizer με frozen όλα τα layers εκτός από FC layer.
optimizer=optim.Adam(
    list(model.fc.parameters())+
    list(model.layer3.parameters())+
    list(model.layer4.parameters()),
    lr=0.001)

In [28]:
#Ξεκινάμε το training loop
#αρχίκά ορίζουμε τον αριθμό των επαναλήψεων
num_epoch=10
best_accuracy=0.0  #Ορίζουμε το καλύτερο accuracy που θα μας δ΄ώσει το μοντέλο μας ώστε να το αποθηκεύσουμε σε μια τιμή 

#Γράφουμε την λούπα για τις επαναλήψεις που θέλουμε αν τρέξει ο αλγόρυθμός μας
for epoch in range(num_epoch):
    model.train() #Ορίζουμε το μοντ΄έλο σε trainning mode
    running_loss=0.0 #Καθώς και το αρχικό loss που θα έχει

    #Και τ΄ώρα μέσα σε αυτό το loop ορίζουμε την λούπα που θα περνάει από όλα αυτα τα batches 
    for inputs,labels in train_loader:   #Για κάθε input μαζί με την ετικέτα του Label θα 
        inputs=inputs.to(device)  #Βάζουμε inputs τα μέσα στο device
        labels=labels.to(device)  #Βάζουμε labels τα μέσα στο device

        optimizer.zero_grad()     #Μηδενίζουμε τα accumulates (gradients) τα οποία συσσωρεύονται από κάθε επανάληψη batch.
        outputs=model(inputs)     #Περνάμε όλα τα inputs από το μοντέλο, 
        loss=criterion(outputs,labels)  #Υπολογίζουμε το Loss,δηλαδή συγκρίνει πόσα από τα prediction (outputs) μοιάζουν με τα labes (πραγματικές τιμές)
        loss.backward()  #Εφαρμογή της backward pass.Αυτό υπολογίζει τα gradients για όλες τις trainable παραμέτρους
        optimizer.step()  #Με τον τρόπο αυτό ενημερώνει τα βάρη (weights) που υπολόγισε η backward().
        running_loss += loss.item()  #Με το item() παίρνουμε τον αριθμό από το loss προσθέτοντας όλα τα loss από τα batches.
    avg_loss = running_loss / len(train_loader)  #Βρίσκουμε τον μέσο τών συνολικών loss 
    print(f'Epoch {epoch+1}/{num_epoch}, Loss: {avg_loss:.4f}') #Εμφανίζουμε το loss μετά από κάθε epoch

    #Evaluation Model.Εδώ τώρα θα ορίσουμε το μοντέλο να λειτουργεί σε evaluation mode.
    model.eval()
    test_loss=0.0   #ορίζουμε το loss μηδέν ΄ώστε να ξεκινήσει να μετράει από αυτό
    correct=0.0     #Υπολογίζει τις σωστές απαντήσεις
    total=0.0       #Μετράει τις συνολικές απαντήσεις

    #Για να φτιάξουμε το Confussion matrix μπορούμε να χρησιμοποιήσουμε την Evaluation Loop.Θα πρέπει όμως να ορίσουμε τα εξής
    all_predictions=[] #Μια κενή λίστα για όλες τις προβλέψεις
    all_targets=[] #Μια κενή λίστα για όλες τις πραγματικές τιμές

#Evaluation Loop
    
    with torch.no_grad():
        for inputs,labels in test_loader: #Για κάθε input και label του test dataset
            inputs=inputs.to(device)   #Περνάμε τα inputs στο device για να βρίσκονται στο ίδιο περιβάλλον με τα train data
            labels=labels.to(device)   #Περνάμε τα labels στο device για να βρίσκονται στο ίδιο περιβάλλον με τα train data 
                                        #Το inputs κρατάει batch_size, channels, height, width και το labels τους κωδικούς κλάσεων ]0,1,2,0,1,...]
            outputs=model(inputs)    #Κάνουμε τις προβλέψεις για τα δεδομένα μας
            loss=criterion(outputs,labels)  #Υπολογίζουμε το loss,δηλαδή πόσες από τις προβλέψεις είναι σωστές
            test_loss += loss.item()     #Κρατάμε το loss κάθε φορά μέσω του .item() και το προσθέτουμε στο test_loss
            _, predicted = torch.max(outputs,dim=1)  #Κρατάμε μόνο τα indices και όχι τα values που μας επιστρέφει η values,indices=torch.max(outputs,dim=1)

            all_predictions.extend(predicted.cpu().numpy())  #Περνάμε όλες τις ετικκέτες που έχουμε προβλέψει στο all_prediction
            all_targets.extend(labels.cpu().numpy())  #Περνάμε όλες τις πραγματικές ετικκέτες στο all_targets
        #Το .extend ουσιαστικά περνάει επιτόπου τις τιμές σε μια λίστα (np) αφού πρώτα τις αποθηκεύσει στο .gpu()
            #Το .append θα περνουσε μόνο μια τείκή τιμή στην λίστα,ενώ εμείς θέλουμε να περάσει κάθε μια τιμή(εικόνα)
                
            correct += (predicted==labels).sum().item()  #Πα΄ίρνουμε το άθροισμα των σωστών απαντήσεων
            total += labels.size(0)  #Προσθέτουμε το συνολικό total δηλαδή πόσες εικόνες έχει αυτό το batch
    avg_test_loss = test_loss / len(test_loader)
    accuracy = 100 * correct/total  #Υπολογίζουμε το accutacy επί τις 100
    print(f'Test Loss: {avg_test_loss:.4f}, Test Accuracy: {accuracy:.2f}%\n')

#Save best Accouracy
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save(model.state_dict(), 'RN50_NotWL_TK-Means50.pth')
        #torch.save(model.state_dict(), 'best_resnet18_waste_density.pth')
        print(f'New best model saved! Accuracy: {best_accuracy:.2f}%\n')
    

    #Δημιουργούμε τον confusion matrix 
    cm=confusion_matrix(all_targets,all_predictions) #Με την μέθοδο confusion_matrix δημιουργούμε ton cm μας.Πρώτα βάζουμε τα (true_values,predicted_values)
    print("\nConfusion Matrix:")
    print(cm)

Epoch 1/10, Loss: 0.5547
Test Loss: 0.9374, Test Accuracy: 69.44%

New best model saved! Accuracy: 69.44%


Confusion Matrix:
[[158  14  44]
 [  2 110 115]
 [ 10   2 157]]
Epoch 2/10, Loss: 0.4378
Test Loss: 0.6311, Test Accuracy: 77.12%

New best model saved! Accuracy: 77.12%


Confusion Matrix:
[[184   5  27]
 [ 19 136  72]
 [ 16   1 152]]
Epoch 3/10, Loss: 0.3792
Test Loss: 0.4044, Test Accuracy: 85.78%

New best model saved! Accuracy: 85.78%


Confusion Matrix:
[[174  11  31]
 [ 10 201  16]
 [ 15   4 150]]
Epoch 4/10, Loss: 0.3189
Test Loss: 0.3481, Test Accuracy: 87.25%

New best model saved! Accuracy: 87.25%


Confusion Matrix:
[[181   8  27]
 [  6 207  14]
 [ 17   6 146]]
Epoch 5/10, Loss: 0.3212
Test Loss: 0.4787, Test Accuracy: 85.46%


Confusion Matrix:
[[188   9  19]
 [  5 197  25]
 [ 26   5 138]]
Epoch 6/10, Loss: 0.3322
Test Loss: 0.4010, Test Accuracy: 85.29%


Confusion Matrix:
[[185  11  20]
 [  8 194  25]
 [ 23   3 143]]
Epoch 7/10, Loss: 0.3110
Test Loss: 0.4147, Test

In [39]:
def predict_confidence(image_path):
    model.load_state_dict(torch.load('RN50_NotWL_TK-Means50.pth'))
    model.eval()
    
    image = Image.open(image_path).convert('RGB')
    image = transform_test(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(image)
        probabilities = F.softmax(outputs, dim=1)
        confidence, predicted_class = torch.max(probabilities, dim=1)
        
        prob=confidence.item()
        predicted=predicted_class.item()
        raw_confidence=int(prob*100)

        if predicted == 0:      
            confidence_score = max(85, raw_confidence)
        elif predicted == 1:   
            confidence_score = max(65, min(84, raw_confidence))
        else:                   
            confidence_score = min(64, raw_confidence)
        
        class_names = {0: "High", 1: "Low", 2: "NoWaste"}
        #predicted = predicted_class.item()
        #confidence_score = confidence_map[predicted]

        print(f"Κατηγορία: {class_names[predicted]}")
        print(f"Confidence: {confidence_score}%")
        print(f"Raw Softmax: {raw_confidence}%")
        print("---")
        
    
    return class_names[predicted], confidence_score


# Δοκιμή
class_name, confidence = predict_confidence("C:/Users/panag/Desktop/full.avif")
print(f"Κατηγορία: {class_name}, Confidence: {confidence}%")

Κατηγορία: Low
Confidence: 84%
Raw Softmax: 99%
---
Κατηγορία: Low, Confidence: 84%
